# 🥇 Exploração da Camada Gold — Bolsa Família

Notebook para análise exploratória das tabelas Gold geradas pelo pipeline.

**Fonte:** Portal da Transparência / dados.gov.br  
**Dataset:** Pagamentos Bolsa Família

In [ ]:
import sys
sys.path.insert(0, '/home/jovyan/src')
sys.path.insert(0, '/opt/airflow/src')

import os
os.environ.setdefault('USE_SAMPLE_DATA', 'true')
os.environ.setdefault('MEDALLION_BASE_PATH', '/home/jovyan/data')

import pandas as pd
import warnings
warnings.filterwarnings('ignore')

print('✅ Imports OK')

In [ ]:
# ── SparkSession ──────────────────────────────────────────────────────────
from src.utils.spark_session import get_spark
spark = get_spark('Notebook-Gold')
print(f'Spark {spark.version} iniciado ✅')

In [ ]:
# ── Roda o pipeline completo (Bronze → Silver → Gold) ──────────────────────
from src.bronze.bronze_ingestion import ingest_bronze
from src.silver.silver_transform import transform_silver
from src.gold.gold_aggregations import build_gold

YEAR, MONTH = 2023, 1

print('🟫 Bronze...')
ingest_bronze(spark, YEAR, MONTH, overwrite=True)

print('🥈 Silver...')
transform_silver(spark, YEAR, MONTH, overwrite=True)

print('🥇 Gold...')
tables = build_gold(spark, YEAR, MONTH, overwrite=True)

print('\n✅ Pipeline concluído!')
for name, df in tables.items():
    print(f'  {name}: {df.count():,} linhas')

## 📊 Análise: Transferências por UF

In [ ]:
from src.utils.config import GOLD_PATH
import pyspark.sql.functions as F

df_uf = spark.read.format('delta').load(str(GOLD_PATH / 'transferencias_por_uf_mes'))
pd_uf = df_uf.orderBy(F.col('total_valor_brl').desc()).toPandas()

print('Top 10 UFs por valor total:')
pd_uf[['uf','total_beneficios','total_valor_brl','media_valor_brl']].head(10)

## 🏙️ Análise: Top Municípios

In [ ]:
df_mun = spark.read.format('delta').load(str(GOLD_PATH / 'top_municipios_beneficios'))
pd_mun = df_mun.filter(F.col('ranking_nacional') <= 20).toPandas()

print('Top 20 Municípios:')
pd_mun[['ranking_nacional','nome_municipio','uf','total_beneficios','total_valor_brl']]

## 📈 Série Temporal

In [ ]:
df_serie = spark.read.format('delta').load(str(GOLD_PATH / 'serie_temporal_brasil'))
pd_serie = df_serie.toPandas()

print('Série Temporal Brasil:')
pd_serie[['ano_mes','total_beneficios','total_valor_brl','variacao_pct']]

In [ ]:
# ── Simples plot com pandas ────────────────────────────────────────────────
try:
    import matplotlib.pyplot as plt
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Plot 1: Valor por UF
    top10 = pd_uf.head(10)
    axes[0].barh(top10['uf'], top10['total_valor_brl'] / 1e6)
    axes[0].set_xlabel('Valor Total (R$ milhões)')
    axes[0].set_title('Top UFs — Valor Total Transferido')
    axes[0].invert_yaxis()

    # Plot 2: Beneficiários por UF
    axes[1].barh(top10['uf'], top10['total_beneficios'])
    axes[1].set_xlabel('Número de Beneficiários')
    axes[1].set_title('Top UFs — Número de Beneficiários')
    axes[1].invert_yaxis()

    plt.tight_layout()
    plt.savefig('/home/jovyan/notebooks/gold_analysis.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('📊 Gráfico salvo: notebooks/gold_analysis.png')
except ImportError:
    print('matplotlib não disponível — instale com: pip install matplotlib')